# IVDC Lesson 8 · Colab 考試速用本**三條規則**1. **Colab 沒有 webcam、也沒有螢幕** → 任何 `cv2.VideoCapture(0)` / `cv2.imshow()` 都會失敗。   本 repo 的腳本已經處理好：`imshow` 自動退回存檔、`--video 檔案.mp4` 可以取代攝影機。2. **不用自己裝 TF / torch / cv2** —— Colab 2026.10 內建 TF 2.21.0、torch 2.11.0+cpu、   torchvision 0.26.0+cpu、opencv 5.0.0、tensorflow-hub 0.16.1。只差 `ultralytics`（YOLO）。3. **`setuptools` 不用降版** —— Colab 是 80.10.2（< 81），`pkg_resources` 還在，   `tensorflow_hub` 直接可以 import。（在自己電腦上才需要 `setuptools<81`。）執行順序：往下依序跑，每格都印出可貼進答案的輸出。

## 1. 取得程式碼

In [ ]:
import osif not os.path.isdir("ivdc_handout"):    !git clone -q https://github.com/kalpakjian/ivdc_handout%cd ivdc_handout!ls src bench

## 2. 環境自檢（先跑這格，出問題就不要往下）

In [ ]:
import sys, importlib, platformprint(platform.platform(), "| Python", sys.version.split()[0])for m in ["numpy", "cv2", "torch", "torchvision", "tensorflow", "tensorflow_hub", "matplotlib", "ultralytics"]:    try:        mod = importlib.import_module(m)        print(f"  OK      {m:16s} {getattr(mod, '__version__', '?')}")    except Exception:        print(f"  MISSING {m:16s}")import torchprint("CUDA:", torch.cuda.is_available(), "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))print("\n-- 本 repo 的完整自檢（Colab 沒有相機，所以跳過）--")!python src/00_check_env.py --skip-camera

## 3. 補裝 ultralytics（只有 YOLO 那格需要）

In [ ]:
!pip install -q ultralyticsimport ultralyticsprint("ultralytics", ultralytics.__version__)

## 4. 測試素材（Colab 沒有你的圖，先抓官方的）

In [ ]:
!wget -q -O bus.jpg https://ultralytics.com/images/bus.jpg!ls -la bus.jpg

## 5. 物件偵測 · 單張圖（torchvision，簡報第 5 頁）注意這是 **CPU** 的 torch；要 1.1 FPS 那個對照組就跑 `--model fasterrcnn_resnet50_fpn`。

In [ ]:
!python src/03_torch_object_detection_image.py --image bus.jpg --model ssdlite320 --threshold 0.4

## 6. 物件偵測 · YOLOv8（簡報第 7 頁的四行方案）

In [ ]:
!python src/05_yolo_detection.py --source bus.jpg --imgsz 640

## 7. 物件偵測 · TensorFlow / TF-Hub（簡報第 2 頁）

In [ ]:
!python src/01_tf_object_detection_image.py --image bus.jpg --threshold 0.5

## 8. 「即時偵測」在 Colab 怎麼辦（簡報第 3／6 頁）**沒有 webcam**，所以把攝影機換成影片檔，流程一行都不用改。考試時如果題目要求即時偵測，就自己產一段影片（下面用 webcam 影格合成不行時，直接拿 bus.jpg 做成 2 秒短片）：```python# 沒有 webcam 時，用一張圖合成一段影片當來源import cv2img = cv2.imread("bus.jpg")w = cv2.VideoWriter("clip.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 15.0, (img.shape[1], img.shape[0]))for _ in range(30):    w.write(img)w.release()```

In [ ]:
import cv2img = cv2.imread("bus.jpg")w = cv2.VideoWriter("clip.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 15.0, (img.shape[1], img.shape[0]))for _ in range(30):    w.write(img)w.release()!python src/04_torch_webcam_detection.py --video clip.mp4 --model ssdlite320

## 9. CNN 分類（簡報第 8–25 頁）· 考試時間短，用 1–2 epochs

In [ ]:
!python src/06_tf_cnn_mnist.py --epochs 1 --no-plot

In [ ]:
!python src/07_torch_cnn_mnist.py --epochs 1

## 10. 看結果圖（Colab 沒有視窗，用 display 把存下來的標註圖貼出來）

In [ ]:
from IPython.display import Image, displayimport globfor p in ["out/torch_detection.jpg", "out/tf_detection.jpg"] + sorted(glob.glob("out/yolo/*.jpg"))[:1]:    if glob.glob(p):        print(p); display(Image(p, width=640))

## 考試當天速查：錯誤 → 解法| 症狀 | 原因 | 解法 ||---|---|---|| `cv2.error: ... not implemented ... GTK+ 2.x` | Colab 沒有 GUI backend | 不要 `imshow`；用 `--no-show`，圖存在 `out/`，用 `display(Image(...))` 看 || `[錯誤] 找不到有畫面的攝影機` | Colab 沒有 webcam | 用 `--video clip.mp4`（第 8 格） || `ModuleNotFoundError: ultralytics` | 沒預裝 | 第 3 格 `!pip install -q ultralytics` || `ModuleNotFoundError: No module named 'pkg_resources'` | 只有**自己電腦**的 setuptools ≥ 81 才會；Colab 不會 | 本機：`pip install 'setuptools<81'` || TF-Hub `hub.load` 很慢 / 卡住 | 第一次要下載 ~20 MB | 等它；或改用 torchvision 那條線 || MNIST 下載失敗 | 短暫網路問題 | 重跑該格（Keras 會重試） || `Runtime disconnected` | 閒置 | 考前把筆記本開著，別關分頁 || 想用 GPU 但只有 CPU 版 torch | 執行環境選錯 | 右上「執行環境 → 變更執行環境 → T4 GPU」 |**貼答案要用的數字**：`03` 會印 `person 0.989 / bus 0.985`；`05` 印 `6 個物件、最高 0.87`；`06`／`07` 印 `Test accuracy`。這些在 CPU 上分別約 30 ms、28 ms、20 秒／epoch。